# UTKFace — chỉ A12, bỏ căn ở trọng số của cả C và O

Train mới, không resume. Giữ 3 seed `[42, 2006, 2026]` như bản gốc: **3 train + 3 test**. Mỗi lần Run all tạo thư mục kết quả mới.

Trong notebook gốc, căn nằm ở trọng số lớp dùng chung, không nằm ở toàn bộ C hay O:

$$a_k=\frac{1-\beta}{1-\beta^{n_k}},\qquad w_k=\frac{a_k}{\operatorname{mean}(a)}.$$

$$C=-w_y(1-p_y)^\gamma\log p_y,\quad L=\operatorname{mean}\left[C+\operatorname{stopgrad}(w_yD)O\right].$$

Chỉ thay `w = a.sqrt()` bằng `w = a`. Giữ nguyên O (ordinal BCE), D, focal gamma=2, stop-gradient, dữ liệu, chia 80/10/10, ResNeXt50 ImageNet V2 và các hyperparameter gốc. Không bỏ bình phương trong n_eff hoặc căn trong metric RMSE vì chúng không phải căn của trọng số lớp.

**Cách chạy:** Add Input UTKFace, bật GPU và Internet (hoặc đặt `WEIGHTS_PATH` đến pretrained ImageNet V2), kiểm tra `DATA_ROOT`, chọn **Run all**. Không cần Add Input output cũ. Các seed chạy song song trên GPU 0 và GPU 1 (mỗi GPU một seed); seed thứ ba chạy khi có GPU trống. Khi chỉ còn một seed, một GPU sẽ rảnh. Chọn accelerator **GPU T4 x2** trên Kaggle; microbatch=32 và logical batch=32. Giữ ít nhất hai seed để tận dụng hai GPU; một seed chỉ sử dụng một GPU trong thiết kế này.

Chọn best checkpoint bằng validation QWK (hòa: MAE rồi macro-F1), sau khi train xong mọi seed mới test. Chỉ lưu best.pt để test/suy luận; không lưu optimizer hoặc checkpoint resume. Chạy lại cell cấu hình trước khi train lại.

Output gồm best.pt từng seed, history, diagnostics, dự đoán/metrics train-val-test, CSV, XLSX và all_results.json. MAE/RMSE tính theo **nhóm tuổi**, không phải số năm. Mean/std tính qua các seed; std mẫu không xác định nếu chỉ một seed.

Bản sửa: microbatch cố định 32; sửa lỗi trùng khóa seed khi tổng hợp kết quả. Nếu cần xuất lại báo cáo từ kết quả đã lưu, gọi `export_results(cfg)` với `cfg` đọc từ config.json của thư mục kết quả đó; thao tác này không train hay test lại.


In [13]:
import os, sys, time
from pathlib import Path
from datetime import datetime
from uuid import uuid4
os.environ['OMP_NUM_THREADS']='2'
os.environ['MKL_NUM_THREADS']='2'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
DATA_ROOT='/kaggle/input/datasets/jangedoo/utkface-new/UTKFace'
WEIGHTS_PATH=''  # ImageNet V2 .pth nếu tắt Internet; KHÔNG phải checkpoint A12 cũ.
TRAINING_SEEDS=[42,2006,2026]  # Đổi thành [42] nếu chỉ muốn một seed.
RUN_TAG=datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+uuid4().hex[:6]
OUT='/kaggle/working/A12_no_sqrt_'+RUN_TAG
CODE_DIR=Path('/kaggle/working/A12_no_sqrt_code')
CODE_DIR.mkdir(parents=True,exist_ok=True)
if str(CODE_DIR) not in sys.path: sys.path.insert(0,str(CODE_DIR))
CFG=dict(schema_version='A12_only_no_sqrt_v1',data_root=DATA_ROOT,out=OUT,
    cache_root='/kaggle/temp/A12_no_sqrt_cache',weights_path=WEIGHTS_PATH,
    training_seeds=TRAINING_SEEDS,split_seed=42,fold_index=None,image_size=224,
    dropout=0.2,freeze_bn_stats=True,deterministic=True,batch_size=32,
    microbatch=32,eval_batch_size=64,learning_rate=3e-4,weight_decay=1e-4,
    beta=0.999,gamma=2.0,epochs=50,patience=10,gradient_clip=5.0,
    timing_repeats=20,evaluate_train_at_best=True,gpu_ids=[0,1],log_poll_seconds=1.0,smoke_testing=False)
print('Fresh A12 runs:',len(TRAINING_SEEDS),'| Output:',OUT)


Fresh A12 runs: 3 | Output: /kaggle/working/A12_no_sqrt_20261028_131647_97d1cf


In [14]:
import importlib.util, subprocess
needed={'numpy':'numpy','pandas':'pandas','PIL':'Pillow','sklearn':'scikit-learn',
        'matplotlib':'matplotlib','tqdm':'tqdm','psutil':'psutil','openpyxl':'openpyxl'}
missing=[pkg for mod,pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
import torch, torchvision, psutil
if not torch.cuda.is_available(): raise RuntimeError('Bật GPU trong Kaggle Settings trước khi Run all.')
print('GPUs:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('CPU logical:',psutil.cpu_count(),'RAM GiB:',round(psutil.virtual_memory().total/1024**3,1))

if torch.cuda.device_count()<2:
    raise RuntimeError('Notebook này cần 2 GPU. Chọn GPU T4 x2 trong Kaggle Settings rồi Run all.')


GPUs: ['Tesla T4', 'Tesla T4']
CPU logical: 4 RAM GiB: 31.3


In [15]:
%%writefile /kaggle/working/A12_no_sqrt_code/data.py

"""Chuẩn bị UTKFace một lần: kiểm tra ảnh, bỏ trùng, chia tập, cache uint8 trên đĩa."""

import os, json, hashlib, shutil, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split, StratifiedKFold
from tqdm.auto import tqdm

LABELS = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70+"]


def atomic_json(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def fingerprint(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def inspect_image(args):
    file, root = args
    try:
        token = file.name.split("_")[0]
        if not token.isdigit():
            raise ValueError("Tên không có tuổi nguyên hợp lệ")
        age = int(token)
        if not 0 <= age <= 116:
            raise ValueError("Tuổi ngoài 0..116")
        with Image.open(file) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            arr = np.asarray(im)
            sha = hashlib.sha256(str(arr.shape).encode() + arr.tobytes()).hexdigest()
        return dict(
            file=str(file.relative_to(root)),
            age=age,
            label=min(age // 10, 7),
            sha256=sha,
            width=arr.shape[1],
            height=arr.shape[0],
            error="",
        )
    except Exception as e:
        return dict(file=str(file.relative_to(root)), error=f"{type(e).__name__}: {e}")


def prepare_data(cfg):
    root = Path(cfg["data_root"])
    out = Path(cfg["out"]) / "data"
    out.mkdir(parents=True, exist_ok=True)
    if not root.is_dir():
        raise FileNotFoundError(f"Không thấy UTKFace tại {root}. Hãy Add Input đúng dataset.")
    files = sorted(
        p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png")
    )
    if not files:
        raise RuntimeError("Không có ảnh trong thư mục dữ liệu.")
    # Kiểm tra inventory mỗi lần; full pixel hash lần đầu hoặc khi danh sách/kích thước file đổi.
    inventory = fingerprint(
        [(str(p.relative_to(root)), p.stat().st_size, p.stat().st_mtime_ns) for p in files]
    )
    invfile = out / "inventory.json"
    if (
        invfile.exists()
        and json.loads(invfile.read_text()).get("fingerprint") == inventory
        and (out / "audit.csv").exists()
    ):
        audit = pd.read_csv(out / "audit.csv", keep_default_na=False)
    else:
        workers = max(1, min(4, os.cpu_count() or 1))
        with ThreadPoolExecutor(max_workers=workers) as pool:
            audit = pd.DataFrame(
                tqdm(
                    pool.map(inspect_image, ((f, root) for f in files)),
                    total=len(files),
                    desc="Kiểm tra và hash ảnh",
                    mininterval=2,
                    miniters=max(1, len(files) // 100),
                )
            )
        audit.to_csv(out / "audit.csv", index=False)
        atomic_json({"fingerprint": inventory}, invfile)
    bad = audit[audit.error != ""]
    bad.to_csv(out / "rejected.csv", index=False)
    valid = audit[audit.error == ""].copy()
    # audit.csv có ô trống của ảnh hỏng: pandas có thể lưu tuổi thành "12.0".
    # Đọc lại phải chuyển chuỗi số sang numeric trước, nếu không resume sẽ lỗi.
    valid[["age", "label"]] = (
        valid[["age", "label"]].apply(pd.to_numeric, errors="raise").astype(np.int64)
    )
    inconsistent = valid.groupby("sha256").age.nunique()
    conflict_ids = inconsistent[inconsistent > 1].index
    valid[valid.sha256.isin(conflict_ids)].to_csv(out / "conflicting_duplicates.csv", index=False)
    clean = valid[~valid.sha256.isin(conflict_ids)].copy()
    duplicates = clean[clean.duplicated("sha256", keep=False)]
    duplicates.to_csv(out / "duplicates.csv", index=False)
    clean = clean.drop_duplicates("sha256", keep="first").sort_values("file").reset_index(drop=True)
    clean["cache_index"] = np.arange(len(clean))
    counts = clean.label.value_counts().reindex(range(8), fill_value=0)
    if (counts < 20).any():
        raise ValueError(f"Có lớp <20 ảnh; cần kiểm tra dữ liệu trước khi chia: {counts.to_dict()}")
    indices = np.arange(len(clean))
    labels = clean.label.to_numpy()
    seed = cfg["split_seed"]
    fold = cfg.get("fold_index")
    if fold is None:
        trainval, test = train_test_split(
            indices, test_size=0.1, random_state=seed, stratify=labels
        )
        train, val = train_test_split(
            trainval, test_size=1 / 9, random_state=seed + 1, stratify=labels[trainval]
        )
    else:
        if fold not in range(5):
            raise ValueError("fold_index phải là None hoặc 0..4.")
        dev, test = list(
            StratifiedKFold(5, shuffle=True, random_state=seed).split(indices, labels)
        )[fold]
        train, val = train_test_split(
            dev, test_size=0.125, random_state=seed + 1, stratify=labels[dev]
        )
    clean["split"] = ""
    for name, ids in [("train", train), ("val", val), ("test", test)]:
        clean.loc[ids, "split"] = name
    for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not set(clean.loc[clean.split == a, "sha256"]) & set(
            clean.loc[clean.split == b, "sha256"]
        )
    content_fp = fingerprint(clean[["file", "age", "label", "sha256"]].to_dict("records"))
    split_fp = fingerprint(clean[["sha256", "label", "split"]].to_dict("records"))
    old_report=out/'split_report.json'
    if old_report.exists() and json.loads(old_report.read_text())['split_fingerprint']!=split_fp:
        raise ValueError('Dataset hoặc split khác output cũ. Không ghi đè split; dùng OUT mới.')
    for name in ["train", "val", "test"]:
        clean[clean.split == name].to_csv(out / f"{name}.csv", index=False)
    clean.to_csv(out / "manifest.csv", index=False)
    # Cache nằm ngoài thư mục kết quả: không đóng gói lại dataset ảnh vào ZIP kết quả.
    cache_dir = Path(cfg["cache_root"])
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_key = fingerprint([content_fp, cfg["image_size"], "RGB_bilinear_v1"])[:20]
    cache = cache_dir / f"images_{cache_key}.npy"
    meta = cache.with_suffix(".json")
    shape = (len(clean), cfg["image_size"], cfg["image_size"], 3)
    required = int(np.prod(shape)) + 4096
    rebuild = True
    if cache.exists() and meta.exists():
        try:
            mm = np.load(cache, mmap_mode="r")
            rebuild = mm.shape != shape or mm.dtype != np.uint8
            del mm
        except Exception:
            rebuild = True
    if rebuild:
        if shutil.disk_usage(cache_dir).free < required + 2 * 1024**3:
            raise OSError("Không đủ đĩa để tạo cache và chừa 2 GiB cho checkpoint.")
        tmp = cache.with_name(cache.stem + ".partial.npy")
        mm = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=shape)

        def load(row):
            with Image.open(root / row.file) as im:
                im = (
                    ImageOps.exif_transpose(im)
                    .convert("RGB")
                    .resize((cfg["image_size"],) * 2, Image.Resampling.BILINEAR)
                )
                return np.asarray(im, dtype=np.uint8)

        with ThreadPoolExecutor(max_workers=max(1, min(4, os.cpu_count() or 1))) as pool:
            rows = list(clean.itertuples(index=False))

            def bounded_images():
                for start in range(0, len(rows), 128):
                    yield from pool.map(load, rows[start : start + 128])

            iterator = bounded_images()
            for i, arr in enumerate(
                tqdm(
                    iterator,
                    total=len(clean),
                    desc="Cache ảnh uint8",
                    mininterval=2,
                    miniters=max(1, len(clean) // 100),
                )
            ):
                mm[i] = arr
                if i % 512 == 0:
                    mm.flush()
        mm.flush()
        del mm
        os.replace(tmp, cache)
        atomic_json(
            {"content_fingerprint": content_fp, "shape": list(shape), "dtype": "uint8"}, meta
        )
    report = {
        "inventory_fingerprint": inventory,
        "content_fingerprint": content_fp,
        "split_fingerprint": split_fp,
        "source_files": len(files),
        "valid_files": len(valid),
        "rejected": len(bad),
        "conflicting_files": int(valid.sha256.isin(conflict_ids).sum()),
        "duplicate_groups": int(duplicates.sha256.nunique()),
        "redundant_removed": int(len(valid) - valid.sha256.isin(conflict_ids).sum() - len(clean)),
        "retained": len(clean),
        "cache_path": str(cache),
        "cache_gib": required / 1024**3,
        "split_seed": seed,
        "fold_index": fold,
        "class_labels": LABELS,
        "split_counts": pd.crosstab(clean.split, clean.label)
        .reindex(columns=range(8), fill_value=0)
        .to_dict("index"),
        "identity_limitation": "Pixel-disjoint, không phải person-disjoint; UTKFace không cung cấp subject ID đáng tin cậy trong tên file.",
    }
    atomic_json(report, out / "split_report.json")
    return report


def epoch_batches(frame, batch_size, seed, balanced=False):
    """Danh sách cache_index, thứ tự tái lập độc lập với trạng thái model/loss."""
    rng = np.random.default_rng(seed)
    ids = frame.cache_index.to_numpy()
    if balanced:
        raise ValueError("Bản E-only sử dụng cùng sampler ngẫu nhiên cho mọi phương pháp.")
    ids = rng.permutation(ids)
    return [ids[i : i + batch_size] for i in range(0, len(ids), batch_size)]

Overwriting /kaggle/working/A12_no_sqrt_code/data.py


In [16]:
%%writefile /kaggle/working/A12_no_sqrt_code/models.py

"""Cùng ResNeXt50 và head softmax 8 lớp cho mọi kịch bản E."""

import torch
from torch import nn
from torch.utils.checkpoint import checkpoint
from torchvision.models import resnext50_32x4d
import torch.nn.functional as F


class OrdinalModel(nn.Module):
    def __init__(self, cfg, family):
        super().__init__()
        self.family = family
        self.K = 8
        if cfg.get("smoke_testing", False):
            self.encoder = nn.Sequential(
                nn.Conv2d(3, 8, 3, 2, 1), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten()
            )
            dim = 8
        else:
            self.encoder = resnext50_32x4d(weights=None)
            state = torch.load(cfg["weights_path"], map_location="cpu", weights_only=True)
            self.encoder.load_state_dict(state, strict=True)
            dim = self.encoder.fc.in_features
            self.encoder.fc = nn.Identity()
        self.dropout = nn.Dropout(cfg["dropout"])
        self.head = nn.Linear(dim, 8)
        self.freeze_bn = cfg["freeze_bn_stats"]

    def train(self, mode=True):
        super().train(mode)
        if mode and self.freeze_bn:
            for module in self.encoder.modules():
                if isinstance(module, nn.modules.batchnorm._BatchNorm):
                    module.eval()
        return self

    def forward(self, x, microbatch=None):
        microbatch = microbatch or len(x)
        if self.training and microbatch < len(x):
            # Giữ nguyên logical batch cho hoán đổi lambda.
            # Checkpoint tính lại encoder khi backward để giảm activation memory.
            feat = torch.cat(
                [checkpoint(self.encoder, c, use_reentrant=False) for c in x.split(microbatch)], 0
            )
        else:
            feat = self.encoder(x)
        feat = self.dropout(feat)
        return {"logits": self.head(feat)}


def preprocess(batch, device, training, seed):
    """Chuyển uint8 lên GPU rồi augmentation. CPU không resize/augment trong train."""
    x = torch.from_numpy(batch).to(device, non_blocking=False).permute(0, 3, 1, 2).float() / 255.0
    if training:
        gen = torch.Generator(device=device).manual_seed(seed)
        B = len(x)
        rand = lambda shape: torch.rand(shape, device=device, generator=gen)
        flip = rand((B, 1, 1, 1)) < 0.5
        x = torch.where(flip, x.flip(3), x)
        angle = (rand((B,)) - 0.5) * (20 * torch.pi / 180)
        scale = 0.90 + 0.10 * rand((B,))
        theta = torch.zeros((B, 2, 3), device=device)
        theta[:, 0, 0] = scale * angle.cos()
        theta[:, 0, 1] = -scale * angle.sin()
        theta[:, 1, 0] = scale * angle.sin()
        theta[:, 1, 1] = scale * angle.cos()
        theta[:, :, 2] = (rand((B, 2)) - 0.5) * 0.08
        grid = F.affine_grid(theta, x.shape, align_corners=False)
        x = F.grid_sample(x, grid, mode="bilinear", padding_mode="reflection", align_corners=False)
        brightness = 0.9 + 0.2 * rand((B, 1, 1, 1))
        contrast = 0.9 + 0.2 * rand((B, 1, 1, 1))
        mean = x.mean((2, 3), keepdim=True)
        x = ((x - mean) * contrast + mean) * brightness
        x = x.clamp(0, 1)
    mean = x.new_tensor([0.485, 0.456, 0.406])[None, :, None, None]
    std = x.new_tensor([0.229, 0.224, 0.225])[None, :, None, None]
    return ((x - mean) / std).contiguous()

Overwriting /kaggle/working/A12_no_sqrt_code/models.py


In [17]:
%%writefile /kaggle/working/A12_no_sqrt_code/metrics.py

"""Metric theo 8 chỉ số nhóm tuổi, KHÔNG phải sai số tuổi tính bằng năm."""

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    precision_recall_fscore_support,
    confusion_matrix,
    matthews_corrcoef,
    roc_auc_score,
    average_precision_score,
)


def finite_or_none(x):
    return float(x) if np.isfinite(x) else None


def compute_metrics(y, pred, probs=None):
    y = np.asarray(y, dtype=int)
    pred = np.asarray(pred, dtype=int)
    K = 8
    cm = confusion_matrix(y, pred, labels=np.arange(K))
    n = cm.sum()
    pr, re, f1, support = precision_recall_fscore_support(
        y, pred, labels=np.arange(K), zero_division=0
    )
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = n - tp - fp - fn
    present = support > 0
    result = {
        "n": int(len(y)),
        "accuracy": float(accuracy_score(y, pred)),
        "qwk": finite_or_none(cohen_kappa_score(y, pred, labels=np.arange(K), weights="quadratic")),
        "group_mae": float(np.abs(y - pred).mean()),
        "balanced_accuracy": float(re[present].mean()),
        "precision_macro": float(pr.mean()),
        "recall_macro": float(re.mean()),
        "f1_macro": float(f1.mean()),
        "f1_weighted": float(np.average(f1, weights=support)),
        "precision_weighted": float(np.average(pr, weights=support)),
        "recall_weighted": float(np.average(re, weights=support)),
        "within1": float((np.abs(y - pred) <= 1).mean()),
        "severe_error_rate": float((np.abs(y - pred) >= 2).mean()),
        "specificity_macro": float(
            np.divide(tn, tn + fp, out=np.zeros(K, dtype=float), where=(tn + fp) > 0).mean()
        ),
        "mcc": finite_or_none(matthews_corrcoef(y, pred)),
        "confusion_matrix": cm.tolist(),
        "per_class": [
            {
                "class": i,
                "support": int(support[i]),
                "precision": float(pr[i]),
                "recall": float(re[i]),
                "f1": float(f1[i]),
                "mae": float(np.abs(pred[y == i] - i).mean()) if support[i] else None,
            }
            for i in range(K)
        ],
        "nll": None,
        "ece15": None,
        "auc_roc_macro": None,
        "auc_pr_macro": None,
    }
    if probs is not None:
        probs = np.asarray(probs, dtype=np.float64)
        if not np.isfinite(probs).all() or (probs < -1e-6).any():
            raise ValueError("Xác suất không hợp lệ.")
        probs = np.maximum(probs, 0)
        probs /= probs.sum(1, keepdims=True)
        result["nll"] = float(-np.log(np.maximum(probs[np.arange(len(y)), y], 1e-15)).mean())
        mode = probs.argmax(1)
        conf = probs.max(1)
        correct = mode == y
        bins = np.minimum((conf * 15).astype(int), 14)
        ece = 0.0
        for b in range(15):
            m = bins == b
            if m.any():
                ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
        result["ece15"] = float(ece)
        result["qwk_argmax"] = finite_or_none(
            cohen_kappa_score(y, mode, labels=np.arange(K), weights="quadratic")
        )
        result["accuracy_argmax"] = float((mode == y).mean())
        mean_pred = np.floor((probs * np.arange(K)).sum(1) + 0.5).clip(0, K - 1).astype(int)
        result["qwk_rounded_expectation"] = finite_or_none(
            cohen_kappa_score(y, mean_pred, labels=np.arange(K), weights="quadratic")
        )
        result["group_mae_rounded_expectation"] = float(np.abs(y - mean_pred).mean())
        onehot = np.eye(K)[y]
        if present.all():
            result["auc_roc_macro"] = float(roc_auc_score(onehot, probs, average="macro"))
            result["auc_pr_macro"] = float(average_precision_score(onehot, probs, average="macro"))
    distance=np.abs(y-pred)
    result.update(group_rmse=float(np.sqrt(np.mean((y-pred)**2))),
        linear_weighted_kappa=finite_or_none(cohen_kappa_score(y,pred,labels=np.arange(K),weights='linear')),
        within2=float((distance<=2).mean()),mean_signed_group_error=float((pred-y).mean()),
        underestimation_rate=float((pred<y).mean()),overestimation_rate=float((pred>y).mean()),
        error_distance_counts=np.bincount(distance,minlength=K).tolist(),
        confusion_matrix_normalized_true=np.divide(cm,support[:,None],out=np.zeros_like(cm,dtype=float),where=support[:,None]>0).tolist(),
        confusion_matrix_normalized_pred=np.divide(cm,cm.sum(0)[None,:],out=np.zeros_like(cm,dtype=float),where=cm.sum(0)[None,:]>0).tolist())
    for i,row in enumerate(result['per_class']):
        row.update(tp=int(tp[i]),fp=int(fp[i]),fn=int(fn[i]),tn=int(tn[i]),
            specificity=float(tn[i]/(tn[i]+fp[i])) if tn[i]+fp[i] else None,
            auc_roc=None,auc_pr=None)
        if probs is not None and 0<support[i]<n:
            row['auc_roc']=float(roc_auc_score((y==i).astype(int),probs[:,i]))
            row['auc_pr']=float(average_precision_score((y==i).astype(int),probs[:,i]))
    if probs is not None:
        result['brier_multiclass']=float(np.square(probs-np.eye(K)[y]).sum(1).mean())
        result['calibration_bins_15']=[dict(bin=b,lower=b/15,upper=(b+1)/15,
            n=int((bins==b).sum()),mean_confidence=float(conf[bins==b].mean()) if (bins==b).any() else None,
            accuracy=float(correct[bins==b].mean()) if (bins==b).any() else None) for b in range(15)]
    return result


def checkpoint_key(m):
    # QWK lớn hơn; hòa tới 1e-8 thì MAE nhỏ hơn, rồi Macro-F1 lớn hơn.
    return (
        round(m["qwk"] if m["qwk"] is not None else -2.0, 8),
        -round(m["group_mae"], 8),
        round(m["f1_macro"], 8),
    )

Overwriting /kaggle/working/A12_no_sqrt_code/metrics.py


In [18]:
%%writefile /kaggle/working/A12_no_sqrt_code/losses.py

"""Các loss và head ordinal. Ký hiệu thống nhất với kịch bản thực nghiệm."""

import math
import torch
from torch import nn
import torch.nn.functional as F


def cb_weights(counts, beta=0.999):
    """Inverse effective number, không lấy căn, sau đó chuẩn hóa mean=1."""
    n = torch.as_tensor(counts, dtype=torch.float64)
    if (n <= 0).any():
        raise ValueError("Tập train phải có đủ mọi lớp.")
    if not 0 <= beta < 1:
        raise ValueError("beta phải thuộc [0,1).")
    a = torch.ones_like(n) if beta == 0 else (1 - beta) / (-torch.expm1(n * math.log(beta)))
    w = a
    return (w / w.mean()).float()


def ordinal_parts(logits, y, weights, gamma):
    """Log-sum-exp tránh log(0), không clamp q làm mất gradient ở mẫu khó."""
    z = logits if logits.dtype == torch.float64 else logits.float()
    K = z.shape[1]
    logp = F.log_softmax(z, dim=1)
    p = logp.exp()
    t = y[:, None] > torch.arange(K - 1, device=z.device)
    log_le = torch.logcumsumexp(logp, dim=1)[:, :-1]
    log_gt = torch.logcumsumexp(logp.flip(1), dim=1).flip(1)[:, 1:]
    ordinal = -torch.where(t, log_gt, log_le).mean(1)
    # Tính trực tiếp tổng xác suất phía sai; tránh 1-q gây triệt tiêu số học.
    log_e = torch.where(t, log_le, log_gt)
    e = log_e.exp()
    # Chuẩn hóa theo max trước khi bình phương: cùng tỷ số nhưng không underflow.
    scaled = torch.exp(log_e - log_e.max(1, keepdim=True).values)
    neff = scaled.sum(1).square() / scaled.square().sum(1)
    neff = neff.clamp(1.0, float(K - 1))
    s = (neff - 1) / (K - 2)
    true_logp = logp.gather(1, y[:, None]).squeeze(1)
    # log(1-p_y) bằng log-sum-exp các lớp khác, ổn định cả khi p_y~1.
    other = logp.masked_fill(F.one_hot(y, K).bool(), -torch.inf)
    delta = torch.logsumexp(other, dim=1).exp()
    focal = delta.pow(gamma)
    wf = weights[y].to(z.dtype) * focal
    cls = -wf * true_logp
    return dict(
        p=p, C=cls, O=ordinal, F=wf, focal=focal, neff=neff, s=s, error_sum=e.sum(1), sbar=e.mean(1)
    )


class LossBank(nn.Module):
    def __init__(self, spec, counts, beta=0.999, gamma=2.0):
        super().__init__()
        self.spec, self.K, self.gamma = spec, len(counts), gamma
        self.register_buffer("weights", cb_weights(counts, beta))

    def terms(self, z, y):
        a = ordinal_parts(z, y, self.weights, self.gamma)
        if self.spec['family'] != 'A12':
            raise ValueError('Notebook này chỉ hỗ trợ A12.')
        wy = self.weights[y].to(z.dtype)
        D = (1 + a['s']) / 2
        cls = a['C']
        lam = (wy * D).detach()
        weighted_O = lam * a['O']
        return cls, weighted_O, lam, a

    def forward(self, out, y, generator=None):
        z = out['logits'] if isinstance(out, dict) else out
        z = z if z.dtype == torch.float64 else z.float()
        cls, weighted_O, lam, a = self.terms(z, y)
        diagnostics = {k:a[k].detach() for k in ('O','neff','s','error_sum')}
        # C is the ACTUAL classification term of this ablation; common_C retained separately.
        diagnostics.update(C=cls.detach(),common_C=a['C'].detach(),lam=lam.detach(),weighted_O=weighted_O.detach())
        return (cls + weighted_O).mean(), diagnostics


def predict(out, family):
    p = out["logits"].float().softmax(1)
    return p.argmax(1), p, 0.0


def sampled_logit_gradients(logits, y, criterion):
    """Actual logit gradients of C and stop-gradient weighted O.
    Zero classification/ordinal gradient yields undefined cosine/ratio -> JSON null.
    """
    z = logits.detach().double().requires_grad_(True)
    cls, ordinal, _, _ = criterion.terms(z,y)
    def grad(term, retain):
        if not term.requires_grad: return torch.zeros_like(z)
        return torch.autograd.grad(term.sum(),z,retain_graph=retain)[0]
    gc = grad(cls,True); go = grad(ordinal,False)
    nc,no = gc.abs().sum(1),go.abs().sum(1)
    tiny=torch.finfo(z.dtype).tiny
    ratio=torch.where(nc>0,no/nc.clamp_min(tiny),torch.full_like(nc,float('nan')))
    sc,so=gc.abs().amax(1),go.abs().amax(1)
    uc,uo=gc/sc.clamp_min(tiny)[:,None],go/so.clamp_min(tiny)[:,None]
    cosine=(uc*uo).sum(1)/(uc.norm(dim=1)*uo.norm(dim=1)).clamp_min(tiny)
    cosine=torch.where((sc>0)&(so>0),cosine.clamp(-1,1),torch.full_like(sc,float('nan')))
    return torch.stack([nc,no,ratio,cosine],1).detach()


Overwriting /kaggle/working/A12_no_sqrt_code/losses.py


In [19]:
%%writefile /kaggle/working/A12_no_sqrt_code/worker.py

"""A12 fresh training; chỉ lưu best checkpoint để đánh giá."""

import os

os.environ.setdefault("OMP_NUM_THREADS", "2")
os.environ.setdefault("MKL_NUM_THREADS", "2")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import sys, json, time, random, traceback, gc, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from data import atomic_json, fingerprint, epoch_batches
from models import OrdinalModel, preprocess
from losses import LossBank, predict, sampled_logit_gradients
from metrics import compute_metrics, checkpoint_key
from bundle import atomic_npz


def seed_all(seed, deterministic=True):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(2)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = deterministic
    torch.use_deterministic_algorithms(deterministic, warn_only=True)


def rng_state():
    return dict(
        python=random.getstate(),
        numpy=np.random.get_state(),
        torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    )


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def save_torch(obj, path):
    path = Path(path)
    tmp = path.with_name(path.name + ".tmp")
    if shutil.disk_usage(path.parent).free < 500 * 1024**2:
        raise OSError("Dung lượng trống <500 MiB; dừng trước khi ghi hỏng checkpoint.")
    with tmp.open("wb") as f:
        torch.save(obj, f)
        f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


def sync(device):
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def amp_context(device):
    return torch.autocast(
        device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"
    )


def make_scaler(enabled):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled, init_scale=1024.0)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled, init_scale=1024.0)


@torch.inference_mode()
def evaluate(model, frame, images, cfg, device, batch_size):
    model.eval()
    preds = []
    probs = []
    ys = []
    ids = []
    cross = 0
    prob_valid = True
    sync(device)
    t0 = time.perf_counter()
    for start in range(0, len(frame), batch_size):
        part = frame.iloc[start : start + batch_size]
        idx = part.cache_index.to_numpy()
        x = preprocess(np.array(images[idx], copy=True), device, False, 0)
        with amp_context(device):
            out = model(x)
        pred, p, c = predict(out, model.family)
        preds.append(pred.cpu().numpy())
        ys.append(part.label.to_numpy())
        ids.append(idx)
        cross += c * len(part)
        if p is None:
            prob_valid = False
        else:
            probs.append(p.cpu().numpy())
    sync(device)
    elapsed = time.perf_counter() - t0
    y = np.concatenate(ys)
    prediction = np.concatenate(preds)
    prob = np.concatenate(probs) if prob_valid else None
    m = compute_metrics(y, prediction, prob)
    m.update(
        threshold_crossing_rate=cross / len(frame),
        evaluation_seconds=elapsed,
        seconds_per_image_including_io=elapsed / len(frame),
        probability_metrics_available=bool(prob_valid),
    )
    arrays = dict(y=y, pred=prediction, cache_index=np.concatenate(ids))
    if prob is not None:
        arrays["probabilities"] = prob
    return m, arrays


def load_best(path):
    # Chỉ đọc best.pt do lượt train hiện tại tạo.
    return torch.load(path, map_location="cpu", weights_only=False)


def run_impl(jobfile):
    job = json.loads(Path(jobfile).read_text())
    cfg = job["cfg"]
    spec = job["spec"]
    run_dir = Path(job["run_dir"])
    run_dir.mkdir(parents=True, exist_ok=True)
    fingerprint_id = job["fingerprint"]
    seed = job["seed"]
    mode = job.get("mode", "train")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if device.type == "cpu" and not cfg.get("smoke_testing", False):
        raise RuntimeError("Cần bật GPU trên Kaggle trước khi train.")
    seed_all(seed, cfg["deterministic"])
    print(f"WORKER START | mode={mode} | seed={seed} | device={device}", flush=True)
    micro = job.get("microbatch", cfg["batch_size"])
    atomic_json(
        {"microbatch": micro, "device": str(device), "torch": torch.__version__, "seed": seed},
        run_dir / "runtime.json",
    )
    donefile = run_dir / "done.json"
    if mode == "train" and any(run_dir.glob("*.pt")):
        raise RuntimeError("Train mới cần thư mục run trống; chạy lại cell cấu hình để tạo OUT mới.")
    manifest = pd.read_csv(Path(cfg["out"]) / "data/manifest.csv")
    train = manifest[manifest.split == "train"]
    val = manifest[manifest.split == "val"]
    split = json.loads((Path(cfg["out"]) / "data/split_report.json").read_text())
    images = np.load(split["cache_path"], mmap_mode="r")
    labels = manifest.set_index("cache_index").label.to_dict()
    counts = train.label.value_counts().reindex(range(8), fill_value=0).to_numpy()
    model = OrdinalModel(cfg, spec["family"]).to(device)
    params = sum(p.numel() for p in model.parameters())
    criterion = LossBank(spec, counts, cfg["beta"], cfg["gamma"]).to(device)
    if mode == "test":
        print("TEST START | n=" + str(len(manifest[manifest.split == "test"])), flush=True)
        lock = Path(cfg["out"]) / "selection_lock.json"
        if not lock.exists():
            raise RuntimeError("Chưa khóa lựa chọn; không được đọc test.")
        locked = json.loads(lock.read_text())
        if job["run_id"] not in locked["test_run_ids"]:
            raise RuntimeError("Run không nằm trong danh sách đánh giá đã khóa.")
        state = load_best(run_dir / "best.pt")
        if state["fingerprint"] != fingerprint_id:
            raise ValueError("Checkpoint không khớp cấu hình.")
        model.load_state_dict(state["model"])
        test = manifest[manifest.split == "test"]
        metrics, arrays = evaluate(
            model, test, images, cfg, device, min(micro, cfg["eval_batch_size"])
        )
        # Đo forward-only sau warmup, không tính decode/CPU và truyền dữ liệu.
        x = preprocess(
            np.array(images[test.cache_index.to_numpy()[: min(32, micro, len(test))]], copy=True),
            device,
            False,
            0,
        )
        with torch.inference_mode(), amp_context(device):
            for _ in range(3):
                model(x)
            sync(device)
            t0 = time.perf_counter()
            for _ in range(cfg.get("timing_repeats", 20)):
                model(x)
            sync(device)
            forward_time = time.perf_counter() - t0
        metrics["forward_timing_batch_size"] = len(x)
        metrics["forward_timing_repeats"] = cfg.get("timing_repeats",20)
        metrics["forward_ms_per_image"] = (
            1000 * forward_time / (len(x) * cfg.get("timing_repeats", 20))
        )
        metrics["forward_images_per_second"] = 1000 / metrics["forward_ms_per_image"]
        metrics["peak_gpu_allocated_gib"] = torch.cuda.max_memory_allocated(device)/1024**3 if device.type=="cuda" else 0.0
        metrics.update(
            run_id=job["run_id"],
            seed=seed,
            method=spec["id"],
            fingerprint=fingerprint_id,
            parameters=params,
        )
        atomic_npz(run_dir / "test_predictions.npz", **arrays)
        atomic_json(metrics, run_dir / "test_metrics.json")
        print(f"TEST DONE | QWK={metrics['qwk']:.5f} | Acc={metrics['accuracy']:.4f} | MAE={metrics['group_mae']:.4f} | F1={metrics['f1_macro']:.4f}", flush=True)
        return
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"]
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, min_lr=1e-6
    )
    scaler = make_scaler(device.type == "cuda")
    epoch0 = 0
    cursor = 0
    best_key = (-2.0, -1e9, -1.0)
    bad_epochs = 0
    history = []
    full_history = []
    partial = {"sum_loss": 0.0, "n": 0, "grad_sum": 0.0, "steps": 0, "correct": 0, "sum_C": 0.0, "sum_O": 0.0, "sum_lambda": 0.0, "sum_weighted_O": 0.0}
    samples = []
    elapsed_prior = 0.0
    peak_prior = 0.0
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    sync(device)
    start_clock = time.perf_counter()

    for epoch in range(epoch0, cfg["epochs"]):
        if bad_epochs >= cfg["patience"]:
            break
        model.train()
        batches = epoch_batches(
            train, cfg["batch_size"], seed + 10007 * epoch, balanced=spec.get("balanced", False)
        )
        tick = max(1, int(np.ceil(len(batches) / 10)))
        epoch_clock = time.perf_counter()
        lr_used = optimizer.param_groups[0]["lr"]
        print(f"EPOCH {epoch+1:02d}/{cfg['epochs']} START | batches={len(batches)} | lr={lr_used:.2e} | microbatch={micro}", flush=True)
        for step in range(cursor, len(batches)):
            idx = batches[step]
            y = torch.tensor([labels[int(i)] for i in idx], device=device, dtype=torch.long)
            x = preprocess(
                np.array(images[idx], copy=True), device, True, seed + epoch * 100000 + step
            )
            before_batch_rng = rng_state()
            # Cho phép AMP scaler giảm và retry cùng batch nếu gradient overflow.
            max_amp_attempts = 20
            for amp_attempt in range(max_amp_attempts):
                optimizer.zero_grad(set_to_none=True)
                with amp_context(device):
                    out = model(x, microbatch=micro)
                gen = torch.Generator(device=device).manual_seed(
                    seed + epoch * 100000 + step + 90000000
                )
                loss, diag = criterion(out, y, generator=gen)
                if not torch.isfinite(loss):
                    raise FloatingPointError(f"Loss không hữu hạn ở epoch {epoch+1}, batch {step}.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                grad = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["gradient_clip"])
                if torch.isfinite(grad):
                    scaler.step(optimizer)
                    scaler.update()
                    break
                if device.type != "cuda" or amp_attempt == max_amp_attempts - 1:
                    raise FloatingPointError(
                        f"Gradient không hữu hạn sau retry ở epoch {epoch+1}, batch {step}."
                    )
                # AMP overflow: giảm scale và chạy lại CÙNG batch, không bỏ qua dữ liệu.
                scaler.update(new_scale=max(scaler.get_scale() / 2, 1e-6))
                restore_rng(before_batch_rng)
                del loss, out, diag
            del before_batch_rng
            partial["sum_loss"] += float(loss.detach()) * len(idx)
            partial["n"] += len(idx)
            partial["grad_sum"] += float(grad)
            partial["steps"] += 1
            partial["correct"] += int((out["logits"].detach().argmax(1) == y).sum())
            for key, name in [("C", "sum_C"), ("O", "sum_O"), ("lam", "sum_lambda"), ("weighted_O", "sum_weighted_O")]:
                partial[name] += float(diag[key].sum())

            if (step + 1) % tick == 0 or step == len(batches) - 1:
                if "lam" in diag:
                    keys = ["C", "O", "neff", "s", "error_sum", "lam", "weighted_O"]
                    grads = sampled_logit_gradients(
                        out["logits"], y, criterion
                    )
                    data = (
                        torch.cat([torch.stack([diag[k] for k in keys], 1), grads], 1).cpu().numpy()
                    )
                    samples.append(
                        np.column_stack(
                            [
                                np.full(len(idx), epoch + 1),
                                np.full(len(idx), step),
                                np.array([labels[int(i)] for i in idx]),
                                data,
                            ]
                        )
                    )
                epoch_elapsed = time.perf_counter() - epoch_clock
                processed_batches = step - cursor + 1
                eta_seconds = epoch_elapsed / processed_batches * (len(batches) - step - 1)
                print(f"EPOCH {epoch+1:02d}/{cfg['epochs']} | {100*(step+1)/len(batches):5.1f}% | batch={step+1}/{len(batches)} | loss={partial['sum_loss']/partial['n']:.5f} | train_acc={partial['correct']/partial['n']:.4f} | lr={lr_used:.2e} | ETA_epoch={eta_seconds:.0f}s", flush=True)
                atomic_json(
                    {
                        "epoch": epoch + 1,
                        "batch": step + 1,
                        "batches": len(batches),
                        "loss": partial["sum_loss"] / partial["n"],
                        "microbatch": micro,
                        "elapsed_seconds": elapsed_prior + time.perf_counter() - start_clock,
                    },
                    run_dir / "progress.json",
                )
            del loss, out, x, y, diag
        cursor = 0
        print(f"EPOCH {epoch+1:02d} VALIDATION START", flush=True)
        val_m, val_arrays = evaluate(model, val, images, cfg, device, min(micro, cfg["eval_batch_size"]))
        key = checkpoint_key(val_m)
        improved = key > best_key
        if improved:
            best_key = key
            bad_epochs = 0
            save_torch(
                {
                    "model": model.state_dict(),
                    "fingerprint": fingerprint_id,
                    "epoch": epoch + 1,
                    "validation": val_m,
                    "spec": spec,
                    "seed": seed,
                },
                run_dir / "best.pt",
            )
            atomic_npz(run_dir / "val_predictions.npz", **val_arrays)
            atomic_json(val_m, run_dir / "val_metrics.json")
        else:
            bad_epochs += 1
        scheduler.step(val_m["qwk"] if val_m["qwk"] is not None else -2.0)
        row = {
            "epoch": epoch + 1,
            "train_loss": partial["sum_loss"] / partial["n"],
            "train_accuracy_online": partial["correct"] / partial["n"],
            "train_C_mean": partial["sum_C"] / partial["n"],
            "train_O_mean": partial["sum_O"] / partial["n"],
            "train_lambda_mean": partial["sum_lambda"] / partial["n"],
            "train_weighted_O_mean": partial["sum_weighted_O"] / partial["n"],
            "grad_norm_before_clip": partial["grad_sum"] / partial["steps"],
            "val_qwk": val_m["qwk"],
            "val_accuracy": val_m["accuracy"],
            "val_nll": val_m["nll"],
            "val_ece15": val_m["ece15"],
            "val_group_mae": val_m["group_mae"],
            "val_f1_macro": val_m["f1_macro"],
            "val_balanced_accuracy": val_m["balanced_accuracy"],
            "learning_rate": lr_used,
            "learning_rate_next": optimizer.param_groups[0]["lr"],
            "epoch_seconds_this_session": time.perf_counter() - epoch_clock,
            "validation_seconds": val_m["evaluation_seconds"],
            "train_images_seen": partial["n"],
            "train_steps": partial["steps"],
            "microbatch": micro,
            "best": improved,
        }
        history.append(row)
        tmp_history = run_dir / "history.csv.tmp"
        pd.DataFrame(history).to_csv(tmp_history, index=False)
        os.replace(tmp_history, run_dir / "history.csv")
        full_history.append({**row, "validation_metrics": val_m, "best_qwk_so_far": best_key[0], "early_stopping_bad_epochs": bad_epochs})
        atomic_json(full_history, run_dir / "history_full.json")
        if samples:
            atomic_npz(
                run_dir / f"diagnostics_epoch_{epoch+1:03d}.npz",
                values=np.concatenate(samples),
                columns=np.array(
                    [
                        "epoch",
                        "batch",
                        "class",
                        "C",
                        "O",
                        "neff",
                        "s",
                        "error_sum",
                        "lambda",
                        "weighted_O",
                        "grad_classification_l1",
                        "grad_ordinal_term_l1",
                        "grad_ratio",
                        "grad_cosine",
                    ]
                ),
            )
        samples = []
        partial = {"sum_loss": 0.0, "n": 0, "grad_sum": 0.0, "steps": 0, "correct": 0, "sum_C": 0.0, "sum_O": 0.0, "sum_lambda": 0.0, "sum_weighted_O": 0.0}
        print(f"EPOCH {epoch+1:02d} END | loss={row['train_loss']:.5f} | val_QWK={val_m['qwk']:.5f} | val_Acc={val_m['accuracy']:.4f} | val_MAE={val_m['group_mae']:.4f} | val_F1={val_m['f1_macro']:.4f} | BEST={best_key[0]:.5f} | improved={improved} | patience={bad_epochs}/{cfg['patience']} | seconds={row['epoch_seconds_this_session']:.1f} | checkpoint=saved", flush=True)
    state = load_best(run_dir / "best.pt")
    model.load_state_dict(state["model"])
    val_m, arrays = evaluate(model, val, images, cfg, device, min(micro, cfg["eval_batch_size"]))
    atomic_npz(run_dir / "val_predictions.npz", **arrays)
    atomic_json(val_m, run_dir / "val_metrics.json")
    sync(device)
    peak = torch.cuda.max_memory_allocated(device) / 1024**3 if device.type == "cuda" else 0.0
    if cfg.get('evaluate_train_at_best',True):
        print('TRAIN BEST CHECKPOINT EVALUATION (no augmentation/dropout)',flush=True)
        train_m, train_arrays = evaluate(model,train,images,cfg,device,min(micro,cfg['eval_batch_size']))
        atomic_json(train_m,run_dir/'train_metrics.json')
        atomic_npz(run_dir/'train_predictions.npz',**train_arrays)
    done = {
        "run_id": job["run_id"],
        "method": spec["id"],
        "family": spec["family"],
        "seed": seed,
        "fingerprint": fingerprint_id,
        "epochs_completed": len(history),
        "best_epoch": state["epoch"],
        "best_validation": val_m,
        "parameters": params,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "optimizer": "AdamW",
        "class_counts": counts.tolist(),
        "class_weights": criterion.weights.detach().cpu().tolist(),
        "training_and_validation_seconds": elapsed_prior + time.perf_counter() - start_clock,
        "gpu_count": int(device.type == "cuda"),
        "peak_gpu_allocated_gib": max(peak, peak_prior),
        "microbatch": micro,
        "logical_batch": cfg["batch_size"],
        "gpu_name": torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU_SMOKE_ONLY",
    }
    atomic_json(done, donefile)
    print(f"TRAIN DONE | best_epoch={done['best_epoch']} | val_QWK={val_m['qwk']:.5f} | epochs={len(history)} | minutes={done['training_and_validation_seconds']/60:.1f}", flush=True)
    return 0


if __name__ == "__main__":
    try:
        code = run_impl(sys.argv[1])
        sys.exit(code or 0)
    except torch.cuda.OutOfMemoryError:
        traceback.print_exc()
        sys.exit(42)
    except Exception:
        traceback.print_exc()
        sys.exit(1)


Overwriting /kaggle/working/A12_no_sqrt_code/worker.py


In [20]:
%%writefile /kaggle/working/A12_no_sqrt_code/bundle.py
import os
import numpy as np
from pathlib import Path

def atomic_npz(path, **arrays):
    path=Path(path)
    tmp=path.with_name(path.name+'.tmp')
    with tmp.open('wb') as f: np.savez_compressed(f, **arrays)
    os.replace(tmp,path)


Overwriting /kaggle/working/A12_no_sqrt_code/bundle.py


In [21]:
%%writefile /kaggle/working/A12_no_sqrt_code/runner.py
import os, sys, json, time, shutil, hashlib, subprocess
from pathlib import Path
import pandas as pd
from data import prepare_data, atomic_json, fingerprint

def ensure_weights(cfg):
    if cfg.get("smoke_testing"):
        return
    import torch
    from torchvision.models import ResNeXt50_32X4D_Weights

    cached=Path(cfg['out'])/'assets'/'resnext50_32x4d-1a0047aa.pth'
    if not cfg.get('weights_path') and cached.is_file(): cfg['weights_path']=str(cached)
    if cfg.get("weights_path"):
        path = Path(cfg["weights_path"])
        if not path.is_file():
            raise FileNotFoundError("Không tìm thấy pretrained checkpoint được chỉ định.")
    else:
        weights = ResNeXt50_32X4D_Weights.IMAGENET1K_V2
        try:
            weights.get_state_dict(progress=True, check_hash=True)
        except Exception as e:
            raise RuntimeError(
                "Không tải được ImageNet weights. Bật Internet trên Kaggle hoặc gán weights_path tới file resnext50_32x4d-1a0047aa.pth đã Add Input. Không tự chuyển sang random init."
            ) from e
        path = Path(torch.hub.get_dir()) / "checkpoints" / Path(weights.url).name
    cached.parent.mkdir(parents=True,exist_ok=True)
    if path.resolve()!=cached.resolve(): shutil.copy2(path,cached)
    path=cached
    cfg["weights_path"] = str(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(2**20), b""):
            h.update(b)
    cfg["weights_sha256"] = h.hexdigest()


def make_job(cfg, spec, seed, common, mode="train"):
    run_id = f"{spec['id']}_s{seed}"
    folder = Path(cfg["out"]) / "runs" / run_id
    ident = fingerprint({"common": common, "spec": spec, "seed": seed})
    return dict(
        cfg=cfg,
        spec=spec,
        seed=seed,
        run_id=run_id,
        run_dir=str(folder),
        fingerprint=ident,
        mode=mode,
        microbatch=cfg.get("microbatch", cfg["batch_size"]),
    )


def dispatch(jobs, cfg, worker_path=None):
    """One process per GPU; next seed starts as soon as a slot becomes free."""
    from collections import deque
    gpu_ids=cfg['gpu_ids']
    if not gpu_ids or len(gpu_ids)!=len(set(gpu_ids)):
        raise ValueError('gpu_ids phải không rỗng và không trùng.')
    worker_path=Path(worker_path) if worker_path else Path(__file__).with_name('worker.py')
    queue=deque(jobs);active={};completed=0
    def relay(item):
        for line in item['reader'].read().splitlines():
            print(f"[GPU {item['gpu']} | {item['job']['run_id']} | {item['job']['mode']}] {line}",flush=True)
    def close(item):
        relay(item);item['reader'].close();item['writer'].close()
    try:
        while queue or active:
            for gpu in gpu_ids:
                if gpu in active or not queue: continue
                job=dict(queue.popleft())
                job['microbatch']=cfg.get('microbatch',cfg['batch_size'])
                job['physical_gpu_id']=gpu
                folder=Path(job['run_dir']);folder.mkdir(parents=True,exist_ok=True)
                jobfile=folder/(job['mode']+'_job.json');atomic_json(job,jobfile)
                env=os.environ.copy()
                env.update(PYTHONHASHSEED=str(job['seed']),PYTHONUNBUFFERED='1',CUDA_VISIBLE_DEVICES=str(gpu))
                logpath=folder/(job['mode']+'.log')
                writer=logpath.open('w');reader=logpath.open('r')
                try:
                    proc=subprocess.Popen([sys.executable,'-u',str(worker_path),str(jobfile)],stdout=writer,stderr=subprocess.STDOUT,env=env)
                except BaseException:
                    reader.close();writer.close();raise
                active[gpu]=dict(process=proc,reader=reader,writer=writer,job=job,gpu=gpu)
                print(f"[START] GPU {gpu}: {job['run_id']} {job['mode']}",flush=True)
            for gpu,item in list(active.items()):
                relay(item)
                code=item['process'].poll()
                if code is None: continue
                close(item);del active[gpu]
                if code:
                    raise RuntimeError(f"{item['job']['run_id']} {item['job']['mode']} failed ({code}); xem log. Không resume: Run all lại để train mới.")
                completed+=1
                print(f"[DONE {completed}/{len(jobs)}] GPU {gpu}: {item['job']['run_id']}",flush=True)
            if active:time.sleep(cfg.get('log_poll_seconds',1.0))
    finally:
        # Stop every child on error/interrupt; do not leave a GPU training silently.
        for item in active.values():
            if item['process'].poll() is None:item['process'].terminate()
        for item in active.values():
            try:item['process'].wait(timeout=10)
            except subprocess.TimeoutExpired:item['process'].kill();item['process'].wait()
            close(item)


def run_all(cfg):
    out=Path(cfg['out'])
    if (out/'runs').exists():
        raise RuntimeError('Không resume. Chạy lại cell cấu hình để tạo OUT mới.')
    seeds=cfg['training_seeds']
    if not seeds or len(seeds)!=len(set(seeds)) or any(type(s) is not int or not 0<=s<2**32 for s in seeds):
        raise ValueError('Seeds phải là các số nguyên không trùng thuộc [0, 2**32).')
    import torch
    if any(type(g) is not int or not 0<=g<torch.cuda.device_count() for g in cfg['gpu_ids']):
        raise RuntimeError('GPU được chọn không tồn tại; hãy bật T4 x2.')
    ensure_weights(cfg)
    report=prepare_data(cfg)
    atomic_json(cfg,out/'config.json')
    scripts=out/'source'; scripts.mkdir(exist_ok=True)
    for p in Path(__file__).parent.glob('*.py'): shutil.copy2(p,scripts/p.name)
    common=fingerprint(dict(cfg=cfg,split=report['split_fingerprint']))
    spec=dict(id='A12',family='A12',formula='C + stopgrad(w_y D) O',weight_power=1.0)
    jobs=[make_job(cfg,spec,seed,common) for seed in seeds]
    atomic_json(dict(method='A12',weight_power=1.0,total_train_runs=len(jobs),training_seeds=seeds),out/'experiment_plan.json')
    dispatch(jobs,cfg)
    atomic_json(dict(test_run_ids=[j['run_id'] for j in jobs],selection_rule='Best validation QWK, then lower group MAE, then higher macro F1; no test-based selection.'),out/'selection_lock.json')
    dispatch([{**job,'mode':'test'} for job in jobs],cfg)
    return export_results(cfg)


def export_results(cfg):
    """Export saved per-seed outputs only; no GPU, training, or resume."""
    out=Path(cfg['out'])
    report=json.loads((out/'data'/'split_report.json').read_text())
    jobs=[dict(seed=seed,run_dir=str(out/'runs'/f'A12_s{seed}')) for seed in cfg['training_seeds']]
    results=[]; rows=[]
    for job in jobs:
        folder=Path(job['run_dir'])
        item=dict(seed=job['seed'],method='A12',weight_power=1.0)
        for name in ['done','train_metrics','val_metrics','test_metrics','history_full']:
            item[name]=json.loads((folder/(name+'.json')).read_text())
        results.append(item)
        row={k:v for k,v in item['test_metrics'].items()
             if k not in {'seed','best_epoch'} and (isinstance(v,(int,float)) or v is None)}
        row.update(seed=job['seed'],best_epoch=item['done']['best_epoch'])
        rows.append(row)
    atomic_json(dict(config=cfg,split=report,results=results,std_note='Sample std (ddof=1); undefined for one seed.'),out/'all_results.json')
    frame=pd.DataFrame(rows)
    summary=frame.drop(columns=['seed','best_epoch'],errors='ignore').agg(['mean','std']).T
    frame.to_csv(out/'test_by_seed.csv',index=False);summary.to_csv(out/'test_mean_std.csv')
    with pd.ExcelWriter(out/'A12_no_sqrt_results.xlsx') as writer:
        frame.to_excel(writer,sheet_name='test_by_seed',index=False)
        summary.to_excel(writer,sheet_name='mean_std')
    return frame,summary


Overwriting /kaggle/working/A12_no_sqrt_code/runner.py


In [22]:
%%writefile /kaggle/working/A12_no_sqrt_code/checks.py

"""Independent formula and gradient checks; no training data or test peeking."""
import json
import torch
from losses import LossBank,ordinal_parts,sampled_logit_gradients


def reference(z,y,spec,w,gamma):
    p=z.softmax(1); py=p[torch.arange(len(y)),y];wy=w[y]
    q=torch.stack([p[:,k+1:].sum(1) for k in range(p.shape[1]-1)],1)
    t=y[:,None]>torch.arange(p.shape[1]-1)
    O=-(t*q.log()+(~t)*torch.log1p(-q)).mean(1)
    e=torch.where(t,1-q,q);neff=e.sum(1).square()/e.square().sum(1)
    s=(neff-1)/(p.shape[1]-2);D=(1+s)/2;f=(1-py)**gamma;C=-wy*f*py.log()
    lam=(wy*D).detach()
    return (C+lam*O).mean()

def run_checks():
    torch.manual_seed(9841);specs=[dict(id="A12",family="A12")];y=torch.tensor([0,2,4,7])
    for spec in specs:
        bank=LossBank(spec,[30,60,500,220,90,110,40,25]).double()
        z=torch.randn(4,8,dtype=torch.float64,requires_grad=True)
        got,_=bank(z,y);want=reference(z,y,spec,bank.weights,bank.gamma)
        torch.testing.assert_close(got,want,rtol=1e-9,atol=1e-9)
        g=torch.autograd.grad(got,z)[0];gref=torch.autograd.grad(want,z)[0]
        torch.testing.assert_close(g,gref,rtol=1e-8,atol=1e-9)
        diag=sampled_logit_gradients(z,y,bank)
        finite=diag[:,3][torch.isfinite(diag[:,3])]
        assert ((finite>=-1)&(finite<=1)).all()
        z2=(torch.randn(4,8)*1000).requires_grad_(True)
        loss,_=bank(z2,y);loss.backward()
        assert torch.isfinite(loss) and torch.isfinite(z2.grad).all()
    counts=torch.tensor([30,60,500,220,90,110,40,25],dtype=torch.float64)
    raw=(1-0.999)/(1-0.999**counts)
    expected=(raw/raw.mean()).float()
    from losses import cb_weights
    torch.testing.assert_close(cb_weights(counts),expected)
    assert not torch.allclose(expected,(raw.sqrt()/raw.sqrt().mean()).float())
    assert not bank.terms(z,y)[2].requires_grad
    return dict(passed=True,method='A12',weight_power=1.0,
                checks=['weights_no_sqrt','forward_reference','gradient_reference','extreme_logits','stop_gradient'])


Overwriting /kaggle/working/A12_no_sqrt_code/checks.py


In [23]:
import importlib, json
for name in ['data','models','metrics','losses','bundle','worker','runner','checks']:
    sys.modules.pop(name,None)
importlib.invalidate_caches()
from checks import run_checks
CFG['implementation_checks']=run_checks()
print(CFG['implementation_checks'])


{'passed': True, 'method': 'A12', 'weight_power': 1.0, 'checks': ['weights_no_sqrt', 'forward_reference', 'gradient_reference', 'extreme_logits', 'stop_gradient']}


In [24]:
from runner import run_all
from IPython.display import display
per_seed, summary=run_all(CFG)
display(per_seed)
display(summary.loc[['qwk','accuracy','group_mae','f1_macro','balanced_accuracy']])
print('Kết quả:',OUT)


Kiểm tra và hash ảnh:   0%|          | 0/23708 [00:00<?, ?it/s]

Cache ảnh uint8:   0%|          | 0/23113 [00:00<?, ?it/s]

[START] GPU 0: A12_s42 train
[START] GPU 1: A12_s2006 train
[GPU 0 | A12_s42 | train] WORKER START | mode=train | seed=42 | device=cuda:0
[GPU 1 | A12_s2006 | train] WORKER START | mode=train | seed=2006 | device=cuda:0
[GPU 0 | A12_s42 | train] EPOCH 01/50 START | batches=578 | lr=3.00e-04 | microbatch=32
[GPU 1 | A12_s2006 | train] EPOCH 01/50 START | batches=578 | lr=3.00e-04 | microbatch=32
[GPU 0 | A12_s42 | train] EPOCH 01/50 |  10.0% | batch=58/578 | loss=1.68184 | train_acc=0.2915 | lr=3.00e-04 | ETA_epoch=108s
[GPU 1 | A12_s2006 | train] EPOCH 01/50 |  10.0% | batch=58/578 | loss=1.37948 | train_acc=0.3680 | lr=3.00e-04 | ETA_epoch=108s
[GPU 0 | A12_s42 | train] EPOCH 01/50 |  20.1% | batch=116/578 | loss=1.50224 | train_acc=0.3257 | lr=3.00e-04 | ETA_epoch=87s
[GPU 1 | A12_s2006 | train] EPOCH 01/50 |  20.1% | batch=116/578 | loss=1.23535 | train_acc=0.3925 | lr=3.00e-04 | ETA_epoch=87s
[GPU 0 | A12_s42 | train] EPOCH 01/50 |  30.1% | batch=174/578 | loss=1.39674 | train_acc=

,n,accuracy,qwk,group_mae,balanced_accuracy,precision_macro,recall_macro,f1_macro,f1_weighted,precision_weighted,...,seconds_per_image_including_io,probability_metrics_available,forward_timing_batch_size,forward_timing_repeats,forward_ms_per_image,forward_images_per_second,peak_gpu_allocated_gib,parameters,seed,best_epoch
0,2312,0.605104,0.916515,0.455017,0.569705,0.593063,0.569705,0.579450,0.607920,0.612940,...,0.001949,True,32,20,1.604858,623.108281,0.305922,22996296,42,30
1,2312,0.585640,0.909230,0.481401,0.548272,0.564422,0.548272,0.554955,0.587582,0.592336,...,0.002138,True,32,20,1.844362,542.192977,0.305922,22996296,2006,19
2,2312,0.601644,0.915964,0.457612,0.574438,0.590579,0.574438,0.581495,0.604456,0.608690,...,0.001945,True,32,20,1.612024,620.338001,0.305922,22996296,2026,21


,mean,std
qwk,0.913903,0.004056
accuracy,0.597463,0.010384
group_mae,0.464677,0.014542
f1_macro,0.571967,0.014768
balanced_accuracy,0.564138,0.013943


Kết quả: /kaggle/working/A12_no_sqrt_20261028_131647_97d1cf


In [26]:
import os
import zipfile

source = "/kaggle/working"
zip_path = "/kaggle/working/kaggle_output.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for root, dirs, files in os.walk(source):
        for file in files:
            path = os.path.join(root, file)

            if path == zip_path:
                continue

            z.write(path, os.path.relpath(path, source))

print("ZIP:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / 1024**2, 2), "MB")

ZIP: /kaggle/working/kaggle_output.zip
Size: 431.65 MB


In [27]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")
display(FileLink("kaggle_output.zip"))

/kaggle/working/kaggle_output.zip